# Review R02 — Classification

**Short on time? Answer only the ★ questions (about 5 minutes each).** The others are optional practice; they come back in the April interview sprint.

One weekly session (~30-45 min). Each notebook's exam is appended under its own heading; once you have answered and Claude has reviewed, this review is closed and the next notebook starts a new one.

Almost no code: model outputs are printed as text, and you *read*, *predict*, or *explain*. Write 2-3 plain bullets per sub-part in **Your answer** (the way you'd say it in an interview). Leave **Review** alone; ask Claude to fill it in when you're done. Role tags: **DS**, **Causal**, **ML**; any question is fair game for any role.

## From notebook 14 — Logistic regression

### Q1 — The coefficient that changed sign
*Role lean: DS · Causal*

Logistic regression on the training set (4,930 customers, 26.5% churned):

```text
                         coef    SE      z      odds ratio
Intercept              -0.638  0.076   -8.39      0.529
Partner[T.Yes]          0.115  0.086    1.33      1.122
Dependents[T.Yes]      -0.619  0.099   -6.22      0.539
PaperlessBilling[T.Yes] 1.062  0.079   13.38      2.893
SeniorCitizen           0.674  0.094    7.16      1.962
tenure                 -0.040  0.002  -21.77      0.961

Churn rate:     Partner = No 32.9%  | Partner = Yes 19.6%
Mean tenure:    Partner = No 23.5   | Partner = Yes 42.1 months
churned ~ Partner alone:  Partner[T.Yes] coef = -0.697
```

(a) Interpret the `Dependents` odds ratio in one sentence a manager would understand. Then convert it to percentage points for a customer whose P(churn) without dependents is 30%.
(b) On its own, having a partner goes with much *lower* churn (coef −0.697). In this model the sign flips to slightly positive and insignificant. Explain what happened.
(c) Is "having a partner causes churn, holding tenure fixed" a sound reading of +0.115? Why or why not?

**Your answer (Q1):**

* 

**Review (Q1):**

_pending_

### Q2 — "Let's move everyone onto annual contracts"
*Role lean: Causal*

Odds ratios for month-to-month vs. two-year (training set):

```text
                                                       month-to-month OR   95% CI         one-year OR
churned ~ Contract                                           24.09                            3.94
churned ~ Contract + tenure + InternetService
          + OnlineSecurity + TechSupport + SeniorCitizen      4.18         [2.82, 6.19]       1.90

Contract         churn rate   mean tenure   customers
Month-to-month     42.9%        18.0          2,704
One year           10.9%        42.3          1,034
Two year            3.0%        56.7          1,192
```

A product manager says: "Month-to-month customers have 4 times the odds of churning even after controlling for everything. If we convert them to two-year contracts, we'll cut their churn odds by 75%."

(a) Why did the odds ratio fall from 24 to 4.2 when the controls were added?
(b) Give two reasons why 4.18 is still not the causal effect of switching a customer's contract.
(c) What would you propose instead to estimate what a contract-conversion offer would actually do?

**Your answer (Q2):**

* 

**Review (Q2):**

_pending_

### ★ Q3 — "Three times as likely"
*Role lean: DS · interview*

Among internet customers in the training set:

```text
InternetService   churn rate   odds
DSL                 19.2%      0.238
Fiber optic         41.8%      0.718

odds ratio (fiber vs DSL) = 3.03
```

A slide says: *"Fiber customers are 3 times as likely to churn as DSL customers."*

(a) What is the slide's mistake? Give the number the sentence should use.
(b) When would the odds ratio and the risk ratio be nearly equal? Why does that fail here?
(c) Rewrite the slide's sentence so it is correct and useful for a retention team.

**Your answer (Q3):**

* 

**Review (Q3):**

_pending_

### ★ Q4 — Why not just run OLS on the 0/1 outcome?
*Role lean: Causal · DS*

Same features, two models on the training set: OLS on `churned` (a "linear probability model") and logistic regression with its **average marginal effects** (the average change in P(churn) per unit, across all customers).

```text
                                  OLS coef   logistic avg. marginal effect
Contract: month-to-month vs 2yr    0.160            0.262
Contract: one year vs 2yr         -0.016            0.116
PaperlessBilling = Yes             0.080            0.078
SeniorCitizen                      0.075            0.051
tenure (per month)                -0.0051          -0.0050
MonthlyCharges (per $)             0.0032           0.0036

OLS fitted values: min -0.228, max 0.712; 660 of 4,930 customers below 0
```

(a) What does it mean that 660 customers have a fitted "probability" below 0, and which customers are they likely to be?
(b) For tenure and paperless billing the two columns nearly agree; for the contract dummies they don't. Why?
(c) Interview version: many applied economists use OLS on 0/1 outcomes on purpose. When is that a reasonable choice, and when would you insist on logistic?

**Your answer (Q4):**

* 

**Review (Q4):**

_pending_

### Q5 — A perfect predictor that breaks the model
*Role lean: ML · DS*

A teammate adds `new_customer = (tenure == 0)` to a logistic regression. In the training set, all **7** new customers have `churned = 0`.

```text
ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals

                  coef        SE         z       P>|z|
Intercept        -3.652      0.218   -16.73     0.000
Month-to-month    1.996      0.194    10.26     0.000
One year          0.868      0.202     4.29     0.000
new_customer    -17.068   6091.647    -0.003    0.998
tenure           -0.034      0.002   -14.11     0.000
MonthlyCharges    0.029      0.002    17.99     0.000

log-likelihood: with new_customer -2150.25 | without -2150.90
predicted P(churn) for the 7 new customers: about 0.000000002 to 0.000000009
```

(a) Why is the coefficient about −17 with an SE of about 6,000? What is the optimizer trying to do?
(b) The p-value is 0.998. Does that mean `new_customer` has no relationship with churn?
(c) Look at what the feature means. Why do tenure-0 customers *never* churn in this data, and what does that tell you about using it (or tenure itself) as a predictor?

**Your answer (Q5):**

* 

**Review (Q5):**

_pending_

## From notebook 15 — Evaluating classifiers

### ★ Q6 — "Just catch 90% of the churners"
*Role lean: DS · interview*

Logistic regression with `addon_count` (not the six flags), test set of 2,113 customers with 561 churners:

```text
cutoff   flagged   churners caught (TP)   false alarms (FP)   recall   precision   specificity
0.50       457            297                  160            0.529      0.650        0.897
0.30       828            429                  399            0.765      0.518        0.743
0.153     1164            505                  659            0.900      0.434        0.575

AUC = 0.837
```

The retention lead says: "Set it so we catch 90% of the churners."

(a) What does that cost, in plain numbers the lead would understand?
(b) What question do you ask before picking any cutoff?
(c) Why is the AUC the same for every row of this table?

**Your answer (Q6):**

* 

**Review (Q6):**

_pending_

### Q7 — Good ranking, bad probabilities
*Role lean: ML · DS*

Two models on the same test set:

```text
                        AUC    accuracy@0.5   top-10% churn rate   share with P >= 0.8   their mean P   actually churned
Naive Bayes, all cols   0.807      0.713             70.6%                 36.7%              0.958           52.5%
Logistic (flags)        0.843      0.801             73.5%                  0.9%              0.815           88.9%
```

(a) Naive Bayes's probabilities are badly off, yet its AUC is only 0.036 lower and its top-10% list is almost as good. How?
(b) Name one task Naive Bayes is still fine for here and one where it would mislead.
(c) How could you fix its probabilities without changing its ranking?

**Your answer (Q7):**

* 

**Review (Q7):**

_pending_

### Q8 — Undersampling to 50/50
*Role lean: ML*

The notebook-14 logistic model refit on an **undersampled** training set: all 1,308 churners plus 1,308 randomly chosen stayers. Same test set.

```text
                        AUC    mean predicted P   actual test churn rate   accuracy@0.5   recall@0.5   intercept
full training set      0.843        0.267                 0.265                0.801         0.537       -1.949
undersampled (50/50)   0.843        0.420                 0.265                0.748         0.800       -0.625

ln(stayers / churners) in the full training set = 1.019
undersampled model with its log-odds shifted down by 1.019: mean predicted P = 0.270
```

(a) Why did the mean predicted probability jump while the AUC didn't move?
(b) What does the 1.019 shift do, and why does it work?
(c) When is undersampling actually worth doing?

**Your answer (Q8):**

* 

**Review (Q8):**

_pending_